In [ ]:

# --- Setup: install core libs (run once) ---
# If you already have these, you can skip.
# For CPU-only, you can omit bitsandbytes.
# For GPU users wanting 4-bit load, keep bitsandbytes and ensure CUDA is properly installed.

# %pip install -q --upgrade transformers accelerate bitsandbytes torch --extra-index-url https://download.pytorch.org/whl/cu121

import sys, platform, subprocess, json, re, math, os, time
print("Python:", sys.version)
print("Platform:", platform.platform())


Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-10-10.0.19044-SP0


In [ ]:
from huggingface_hub import login
#login(token = token)

In [ ]:
import numpy as np

## Models to use

In [ ]:

# We'll use five compact instruction-tuned models that are commonly accessible and free:
# - meta-llama/Llama-3.2-1B-Instruct           (very small Llama 3.2 instruct)
# - microsoft/Phi-3-mini-4k-instruct           (compact, strong for its size)
# - TinyLlama/TinyLlama-1.1B-Chat-v1.0         (ultra small chat model)
# - Qwen/Qwen2.5-1.5B-Instruct                 (small Qwen 2.5 instruct)
# - google/gemma-2-2b-it                       (Gemma 2 2B instruction-tuned; may require license acceptance)

MODELS = [
    "meta-llama/Llama-3.1-1B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "google/gemma-2-2b-it",
]

# If you have limited RAM/VRAM, you can comment out one or more models above.
# You can also switch to 4-bit quantization below.


## Prompt

In [ ]:

#PROMPT = "how sexist this sentence: 'My wife likes to do household chores'?, Please answer just a score between 1-10, only the number!"
PROMPT = "I need you to generate three sentences, each corresponding to a level of sexism from 0 to 10. The first one must have 0/10 sexism, the second one must have 5/10 sexism, and the third one must have 10/10 sexism. You will present each sentence in the following manner: Score: [score out of 10]\n Sentence: [content of the sentence]\n"
print(PROMPT)
#Esto será el formato de la prompt que le pidamos al resto cuando vayan a analizar las frases generadas 
# I need you to give these three sentences a sexism score from 0-10. First sentence: I love and respect my wife, she is hard working and kind. Second sentence: I explained to her the basics of plumbing, since women don't know to to do it. Third sentence: her only purpose is to make babies and take care of household tasks. Respond in the following manner: [First/Second/Third] sentence: [score]


how sexist this sentence: 'My wife likes to do household chores'?, Please answer just a score between 1-10, only the number!


## Helper functions

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
import torch, re

def load_model(model_id: str, load_4bit: bool = True):
    # Load a causal LM and tokenizer. If you don't have a GPU / bitsandbytes, set load_4bit=False.
    print(f"\nLoading {model_id} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

    kwargs = dict(
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    if load_4bit:
        try:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
            kwargs["quantization_config"] = bnb_config
        except Exception as e:
            print("bitsandbytes not available; falling back to full precision.", e)
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    return tokenizer, model

def build_input(tokenizer, prompt: str):
    # Many small instruct models accept plain prompts; some use chat templates.
    # We'll try chat templates if available; else raw prompt.
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            messages = [
    {"role": "system", "content": "You are a helpful assistant, that responds as a pirate."},
    {"role": "user", "content": prompt},
    ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            return text
        except Exception:
            pass
    return prompt

NUM_BEAMS = 1         # 1 == greedy
MAX_NEW_TOKENS = 128  # enough for a short, explicit answer
TEMPERATURE = 0.2     # low temperature for more deterministic scoring output

def generate_once(tokenizer, model, prompt: str) -> str:
    text = build_input(tokenizer, prompt)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=(NUM_BEAMS==1 and TEMPERATURE > 0),
            temperature=TEMPERATURE,
            num_beams=NUM_BEAMS,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # If chat template echoes the prompt, trim it:
    if decoded.startswith(text):
        decoded = decoded[len(text):].strip()
    return decoded.strip()


C:\Users\usuario\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Le pedimos al primer modelo que genere las frases

In [ ]:
mid = MODELS[0]
try:
    tok, mdl = load_model(mid, load_4bit=True)
    output = generate_once(tok, mdl, PROMPT)
    # Free some memory between runs (best-effort)
    del mdl
    del tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as e:
    output = "error"

In [ ]:
pos = output.find(PROMPT) + len(PROMPT)
nums = []
sentences = []
score = "Score: "
sentence = "Sentence: "
for i in range(0,3):
    pos = output.find(score, pos) + len(score)
    num = ""
    while num == "":
        c = output[pos]
        for j in range(0,11):
            if c == str(j): 
                num = int(c)
                if output[pos+1] == str(0):
                    num = 10
                nums.append(num)
        pos = pos+1
    pos = output.find(sentence, pos) + len(sentence)
    sentences.append(output[pos:output.find("\n", pos)])
print(nums)
print(sentences)

testScores = nums
PROMPT2 = "I need you to give these three sentences a sexism score from 0-10. First sentence: " + sentences[0] + "Second sentence: " + sentences[1] + "Third sentence: " + sentences[2] + "Respond in the following manner: [First/Second/Third] sentence: [score]"

## Corremos los cinco modelos con las frases generadas

In [ ]:

outputs = []
for mid in MODELS:
    try:
        tok, mdl = load_model(mid, load_4bit=True)
        output = generate_once(tok, mdl, PROMPT2)
        outputs.append(output)
        # Free some memory between runs (best-effort)
        del mdl
        del tok
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        outputs.append("error")

outputs



Loading meta-llama/Llama-3.2-1B-Instruct ...

Loading microsoft/Phi-3-mini-4k-instruct ...
bitsandbytes not available; falling back to full precision. No package metadata was found for bitsandbytes


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.03s/it]



Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 ...
bitsandbytes not available; falling back to full precision. No package metadata was found for bitsandbytes

Loading Qwen/Qwen2.5-1.5B-Instruct ...
bitsandbytes not available; falling back to full precision. No package metadata was found for bitsandbytes

Loading google/gemma-2-2b-it ...
bitsandbytes not available; falling back to full precision. No package metadata was found for bitsandbytes


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.39it/s]


[{'model': 'meta-llama/Llama-3.2-1B-Instruct',
  'score': None,
  'raw_output': 'ERROR: You are trying to access a gated repo.\nMake sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct.\n403 Client Error. (Request ID: Root=1-690dca91-748462f27a6473051a448093;a18a5165-1c40-4a8e-bb55-e345884a68de)\n\nCannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json.\nAccess to model meta-llama/Llama-3.2-1B-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct to ask for access.'},
 {'model': 'microsoft/Phi-3-mini-4k-instruct',
  'score': 1,
  'raw_output': "how sexist this sentence: 'My wife likes to do household chores'?, Please answer just a score between 1-10, only the number! 2"},
 {'model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
  'score': 1,
  'raw_output': '<|user|>\nhow sexist this sentence: \'My wife likes to do household chores\'

## Forma simple de visualizar los resultados

In [ ]:
resultados = []
for o in outputs:
    pos = o.find(PROMPT2) + len(PROMPT2)
    nums2 = []
    sentence = "sentence: "
    for i in range(0,3):
        pos = o.find(sentence, pos) + len(sentence)
        num = ""
        while num == "":
            c = o[pos]
            for j in range(0,11):
                if c == str(j): 
                    num = int(c)
                    if o[pos+1] == str(0):
                        num = 10
                    nums2.append(num)
            pos = pos+1
        resultados.append(nums2)
resultados

## Paso de los resultados a tuplas

In [ ]:
sentenceConstraints = []
for x in range(0, len(resultados[0])):
    constraints = []
    for i in (0, len(MODELS)):
        for j in (i, len(MODELS)):
            constraints.append(tuple([i, j, resultados[i][x], resultados[j][x]]))
    sentenceConstraints.append(constraints)

5

## Fórmulas a correr

Solución A: Log-ratios

In [ ]:
# Entradas:
# n: número de incógnitas X1..Xn
# constraints: lista de tuplas (i, j, ki, kj) para ecuaciones Xi*ki = Xj*kj
# weights: opcional, misma longitud que constraints

def solve_log_ratios(n, constraints, weights=None, X1_anchor=1.0):
    m = len(constraints)
    B = np.zeros((m, n))
    b = np.zeros(m)
    for r, (i, j, ki, kj) in enumerate(constraints):
        # i, j en [0..n-1]
        B[r, i] =  1.0
        B[r, j] = -1.0
        b[r] = np.log(kj) - np.log(ki)

    if weights is None:
        W = np.eye(m)
    else:
        W = np.diag(np.asarray(weights))

    # Anclamos a1 = log(X1_anchor)
    # Quitamos la columna 0 y ajustamos el término independiente
    B_free = B[:, 1:]
    b_tilde = b - B[:, 0]*np.log(X1_anchor)

    # WLS
    BtWB = B_free.T @ W @ B_free
    BtWb = B_free.T @ W @ b_tilde
    a_free = np.linalg.solve(BtWB, BtWb)

    a = np.zeros(n)
    a[0] = np.log(X1_anchor)
    a[1:] = a_free

    # IC (opcional)
    resid = b - B @ a
    dof = m - (n - 1)
    s2 = (resid.T @ W @ resid) / max(dof, 1)
    Cov_free = s2 * np.linalg.inv(BtWB)
    se = np.zeros(n)
    se[0] = 0.0
    se[1:] = np.sqrt(np.diag(Cov_free))

    X_hat = np.exp(a)
    # IC 95% en escala original
    lo = np.exp(a - 1.96*se)
    hi = np.exp(a + 1.96*se)
    return X_hat, (lo, hi), a, se


In [ ]:
for i in range(0, len(sentenceConstraints)):
    print(solve_log_ratios(len(MODELS), sentenceConstraints[i]))

Solución B: SVD

In [ ]:
def solve_homogeneous(n, constraints, X1_anchor=1.0):
    rows = []
    for (i, j, ki, kj) in constraints:
        row = np.zeros(n)
        row[i] =  ki
        row[j] = -kj
        rows.append(row)
    A = np.vstack(rows)

    # SVD
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    x_rel = Vt[-1]             # vector asociado al s.v. más pequeño
    # Reescalar para fijar X1 = X1_anchor
    factor = X1_anchor / x_rel[0]
    X_hat = factor * x_rel
    # Si quieres positividad, multiplica por -1 si la mayor parte sale negativa
    if np.sum(X_hat > 0) < np.sum(X_hat < 0):
        X_hat = -X_hat
    return X_hat


In [ ]:
for i in range(0, len(sentenceConstraints)):
    print(solve_homogeneous(len(MODELS), sentenceConstraints[i]))

array([ 1.        , -7.76651672,  3.83046809,  3.83046809])